# Download Mosaiks data for the cities in our analysis
This code is derived from the [Mosaiks tutorial on querying Redivis for native resolution grid features](https://github.com/Global-Policy-Lab/mosaiks_tutorials/blob/main/01_querying_redivis.ipynb)

In [1]:
import os, time

import pandas as pd
from matplotlib import pyplot as plt


import redivis

In [2]:
## These variables will not change
user = redivis.user("sdss_data_repository")
dataset = user.dataset("mosaiks:8bqm:v1_0")

In [3]:
## Now, let's run this first query. This will ask us to use a browser

#dataset.list_tables() # This output is hard to read

## Modification of the above so that it is a bit easier to read
[item.name for item in dataset.list_tables()]

['0.1 x 0.1 Deg - Area Weighted',
 '0.1 x 0.1 Deg - Pop weighted',
 '0.25. x 0.25 Deg Grid',
 'mosaiks_2019_planet',
 '0.1 x 0.1 Deg - Pop Weighted - TABLE',
 '0.1 x 0.1 Deg - Area Weighted - TABLE',
 '1 x 1 Deg Grid',
 'Administrative Region Aggregations']

Some of these tables are just used for indexing of uploaded files. There are 3 that may be queried for features:

1. 'mosaiks_2019_planet' : This table contains the global, native resolution features (.01 x .01 degree).
2. 'coarsened_0.1_x_0.1_deg_area_weighted_mosaiks' : This table contains the features coarsend to .1 x .1 degree using area weights.
3. 'coarsened_0.1_x_0.1_deg_pop_weighted_mosaiks' : This table contains the features coarsend to .1 x .1 degree using population weights from GHS-POP.

In this notebook, we're going to query for native resolution features


In [4]:
table_name = "mosaiks_2019_planet" # Table name for native resolution grid features
table = dataset.table(table_name)

### Read the cities we are interested in, and get the bounding box


We use the bounding boxes for each city we generated and loop over them to download Mosaiks data

In [ ]:
import geopandas as gpd

# Read the geojson file
geojson_path = "/data-store/iplant/home/shared/esiil/Innovation_Summit_2026/Group_12/city_bounding_box_half_km.geojson"
gdf = gpd.read_file(geojson_path)

# Display basic info
print(f"Geometry type: {gdf.geometry.type.unique()}")
print(f"Number of features: {len(gdf)}")
print(f"Columns: {gdf.columns.tolist()}")

# Show first few rows
gdf

### Query Redivis for the each city's embeddings

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

output_dir = "../data/city_embeddings/"
os.makedirs(output_dir, exist_ok=True)

# Reproject to lat/lon so geometry.bounds matches the lon/lat columns in the Redivis table
gdf_ll = gdf.to_crs(epsg=4326)

# Trial knobs — bump these up once the pipeline runs end-to-end
# N_FEATURES = 500           # set to 4000 for the full embedding
# TRIAL_CITIES = 1          # set to None to process all cities

# feature_cols = ", ".join(f"X_{i}" for i in range(N_FEATURES))
# select_cols = f"lon, lat, shapeGroup, adm2_shapeID_geoBoundaries, adm1_shapeID_geoBoundaries, {feature_cols}"

def _float32_schema(schema: pa.Schema) -> pa.Schema:
    """Return schema with X_* embedding columns coerced to float32."""
    return pa.schema([
        pa.field(f.name, pa.float32()) if f.name.startswith("X_") and pa.types.is_floating(f.type) else f
        for f in schema
    ])


def _cast_batch(batch: pa.RecordBatch, target_schema: pa.Schema) -> pa.RecordBatch:
    arrays = [
        batch.column(i).cast(target_schema.field(i).type) if batch.schema.field(i).type != target_schema.field(i).type
        else batch.column(i)
        for i in range(batch.num_columns)
    ]
    return pa.RecordBatch.from_arrays(arrays, schema=target_schema)


for idx, row in gdf_ll.iterrows():
    uid = row["UID"]
    out_path = os.path.join(output_dir, f"mosaiks_{uid}.parquet")
    print(f"[{idx+1}/{len(gdf_ll)}] UID={uid}: ", end="", flush=True)

    if os.path.exists(out_path):
        print("already downloaded, skipping")
        continue

    xmin, ymin, xmax, ymax = row.geometry.bounds
    print(f"bounds=({xmin:.4f}, {ymin:.4f}, {xmax:.4f}, {ymax:.4f}) — querying...", flush=True)

    start = time.time()
    query = dataset.query(f"""
        SELECT *
        FROM {table_name}
        WHERE lon > {xmin}
        AND lon < {xmax}
        AND lat > {ymin}
        AND lat < {ymax}
    """)

    batch_iter = iter(query.to_arrow_batch_iterator(progress=True))
    try:
        first = next(batch_iter)
    except StopIteration:
        print(f"  -> empty result ({(time.time()-start)/60:.2f} min)")
        continue

    target_schema = _float32_schema(first.schema)
    n_rows = 0
    tmp_path = out_path + ".tmp"
    with pq.ParquetWriter(tmp_path, target_schema, compression="snappy") as writer:
        writer.write_batch(_cast_batch(first, target_schema))
        n_rows += first.num_rows
        for batch in batch_iter:
            writer.write_batch(_cast_batch(batch, target_schema))
            n_rows += batch.num_rows
    os.replace(tmp_path, out_path)

    print(f"  -> {n_rows} rows in {(time.time()-start)/60:.2f} min, wrote {out_path}", flush=True)


### Output of cell above